# Midiland Colab Training Notebook

This notebook is only for the heavy part: downloading the MIDI dataset, rebuilding the gitignored data folders, and training checkpoints on Colab.

It assumes you want checkpoints saved in Google Drive so they survive Colab disconnects.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/midiland'
%cd $PROJECT_DIR

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os
from pathlib import Path
import json
import torch

print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## Configure paths and dataset

In [ ]:
DATASET_ID = 'drengskapur/midi-classical-music'
REVISION = None

DATA_MIDI = 'data_midi'
DATA_TOKENS = 'data_tokens'
DATA_WINDOWS = 'data_windows_full'

CHECKPOINT_DIR = '/content/drive/MyDrive/midiland_checkpoints'

SEQ_LEN = 2048
STRIDE = 1024
MIN_LEN = 256

BATCH_SIZE = 8
NUM_WORKERS = 2
TOTAL_STEPS = 20000
VAL_FRACTION = 0.01
SAVE_EVERY = 500
EVAL_EVERY = 500
LOG_EVERY = 50

## Optional: set Hugging Face token

Only needed if the dataset requires auth. Leave this cell alone if the dataset is public.

In [ ]:
# os.environ['HF_TOKEN'] = '...'

## 1. Download MIDI files into `data_midi/`

In [ ]:
REV_ARG = f'--revision {REVISION}' if REVISION else ''
!python -m midiland.cli.download_hf_dataset {DATASET_ID} {DATA_MIDI} --mode hardlink {REV_ARG}

## 2. Convert MIDI files into per-song token arrays in `data_tokens/`

In [ ]:
!python -m midiland.cli.preprocess_dataset {DATA_MIDI}/midis {DATA_TOKENS} --steps-per-beat 8 --dtype uint16 --print-every 200

In [ ]:
print(Path(DATA_TOKENS, 'stats.json').read_text())

## 3. Build fixed-length windows in `data_windows_full/`

In [ ]:
!python -m midiland.cli.make_windows {DATA_TOKENS} {DATA_WINDOWS} --seq-len {SEQ_LEN} --stride {STRIDE} --min-len {MIN_LEN} --print-every 100

In [ ]:
print(Path(DATA_WINDOWS, 'stats.json').read_text())

## 4. Train on Colab and save checkpoints to Drive

In [ ]:
!python -m midiland.cli.train_lm {DATA_WINDOWS} --out {CHECKPOINT_DIR} --device cuda --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} --val-fraction {VAL_FRACTION} --steps {TOTAL_STEPS} --eval-every {EVAL_EVERY} --save-every {SAVE_EVERY} --log-every {LOG_EVERY} --amp

## 5. Resume later from the latest checkpoint

`--steps` is the total target step count, not an increment.

Example: if `latest.pt` was saved at step 20000 and you want to continue to 40000, set `RESUME_TO_STEP = 40000`.

In [ ]:
RESUME_TO_STEP = 40000
RESUME_CKPT = f'{CHECKPOINT_DIR}/latest.pt'

In [ ]:
!python -m midiland.cli.train_lm {DATA_WINDOWS} --out {CHECKPOINT_DIR} --device cuda --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} --val-fraction {VAL_FRACTION} --steps {RESUME_TO_STEP} --eval-every {EVAL_EVERY} --save-every {SAVE_EVERY} --log-every {LOG_EVERY} --resume {RESUME_CKPT} --amp